# Assignment 3 — Task 3
## Implementation of the Proposed NLP Architecture

**Students:** Raqib Hayat (NUM-BSCS-2022-40), Abu Bakar (NUM-BSCS-2022-41)

This notebook trains every model from `04_architecture_math.md` end-to-end
on the SentiUrdu-1M dataset and saves test-set predictions for
`03_evaluation.ipynb`.

For each of the two tasks we train seven models:

| Family | Models |
|---|---|
| Classical (TF-IDF) | Logistic Regression, Linear SVM |
| Deep Learning      | Multi-kernel CNN, BiLSTM + attention |
| Transformer        | mBERT, XLM-RoBERTa, Urdu-RoBERTa |

All models share:
- the same 8-step preprocessing pipeline (`preprocessing.py`),
- the same 70/15/15 stratified split (`seed=42`),
- inverse-frequency class weighting in the loss,
- identical metric computation in §10.

**Runtime knob.** Set `TRAIN_SUBSET` near the top of §1 to a smaller value
(e.g. 100_000) if you only have a few minutes; set it to `None` to use
the full corpus.


## 1 — Setup


In [1]:
# --- runtime / path setup -------------------------------------------------
import os, sys, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("PYTHONIOENCODING", "utf-8")
ROOT = os.path.dirname(os.path.abspath("__file__"))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import config
import preprocessing as pp
from config import (
    DATA_CSV, FIG_DIR, MODEL_DIR, RESULTS_DIR, CACHE_DIR, EMBED_DIR,
    SEED, set_seed,
    EMOTION_LABELS, SENTIMENT_LABELS,
    EMOTION_TO_SENTIMENT,
)
set_seed(SEED)
print(f"seed={SEED}   data csv: {DATA_CSV.exists()}")


seed=42   data csv: True


In [2]:
import json, time, gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

print("torch", torch.__version__, "cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")

# ───── runtime knob ─────
TRAIN_SUBSET = None          # set e.g. 200_000 to debug end-to-end quickly
# Transformers can be very compute-heavy; if set, also caps the corpus seen
# by the transformer family only (classical + deep still use full data).
TRANSFORMER_TRAIN_SUBSET = None
# ────────────────────────


torch 2.12.0.dev20260222+cu128 cuda available: True
device: NVIDIA GeForce RTX 5070 Ti (17.1 GB)


## 2 — Load and label the dataset


In [3]:
%%time
raw = pd.read_csv(DATA_CSV, dtype=str, keep_default_na=False, na_values=[""])
raw["clean_text"] = pp.preprocess_series(raw["Text"])
raw["sentiment"]  = raw.apply(pp.build_sentiment_label, axis=1)
raw["emotion"]    = raw["Category"].map(pp.majority_emotion)

# Drop empty cleaned tweets (very few)
raw = raw[raw["clean_text"].str.len() > 0].reset_index(drop=True)
print(f"After dropping empty cleaned text: {len(raw):,} rows")

sentiment_df = raw.dropna(subset=["sentiment"]).reset_index(drop=True).copy()
emotion_df   = raw.dropna(subset=["emotion"]).reset_index(drop=True).copy()
print(f"Sentiment task: {len(sentiment_df):,} labelled rows")
print(f"Emotion task  : {len(emotion_df):,} labelled rows")


After dropping empty cleaned text: 1,046,828 rows
Sentiment task: 532,661 labelled rows
Emotion task  : 532,661 labelled rows
CPU times: total: 44.6 s
Wall time: 44.7 s


In [4]:
# Map text labels to integer ids
from config import SENT2ID, ID2SENT, EMOTION2ID, ID2EMOTION
sentiment_df["label"] = sentiment_df["sentiment"].map(SENT2ID)
emotion_df["label"]   = emotion_df["emotion"].map(EMOTION2ID)

print("Sentiment labels:", dict(sentiment_df["label"].value_counts().sort_index()))
print("Emotion labels  :", dict(emotion_df["label"].value_counts().sort_index()))


Sentiment labels: {0: np.int64(72081), 1: np.int64(1547), 2: np.int64(459033)}
Emotion labels  : {0: np.int64(459033), 1: np.int64(50375), 2: np.int64(2798), 3: np.int64(1837), 4: np.int64(17071), 5: np.int64(1547)}


## 3 — Stratified 70/15/15 split and persist


In [5]:
def stratified_split(df, label_col="label", seed=SEED):
    train, temp = train_test_split(df, test_size=0.30, stratify=df[label_col], random_state=seed)
    val, test  = train_test_split(temp, test_size=0.50, stratify=temp[label_col], random_state=seed)
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

splits = {}
for task, df in [("sentiment", sentiment_df), ("emotion", emotion_df)]:
    tr, va, te = stratified_split(df)
    splits[task] = {"train": tr, "val": va, "test": te}
    for name, sub in splits[task].items():
        sub[["clean_text", "label"]].to_parquet(CACHE_DIR / f"{task}_{name}.parquet", index=False)
    print(f"{task}: train={len(tr):,}  val={len(va):,}  test={len(te):,}")


sentiment: train=372,862  val=79,899  test=79,900
emotion: train=372,862  val=79,899  test=79,900


In [6]:
if TRAIN_SUBSET:
    for task in splits:
        tr = splits[task]["train"]
        sub = tr.groupby("label", group_keys=False).apply(
            lambda g: g.sample(min(len(g), max(1, int(len(g) * TRAIN_SUBSET / len(tr)))),
                               random_state=SEED)
        )
        splits[task]["train"] = sub.reset_index(drop=True)
        print(f"  {task}: subsampled train -> {len(sub):,}")


## 4 — Classical: TF-IDF + Logistic Regression and Linear SVM

Section 4.1 / 4.2 of the math doc. We use the *same* TF-IDF vectoriser
for both classical models on each task; only the classifier head changes.


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
import joblib

def run_classical(task: str):
    tr, va, te = splits[task]["train"], splits[task]["val"], splits[task]["test"]
    labels = sorted(tr["label"].unique())
    print(f"\n=== {task.upper()} — TF-IDF + LR/SVM ===")

    vec_path = MODEL_DIR / f"tfidf_{task}.joblib"
    if vec_path.exists():
        vec = joblib.load(vec_path)
        Xtr = vec.transform(tr["clean_text"])
    else:
        vec = TfidfVectorizer(**config.TFIDF)
        Xtr = vec.fit_transform(tr["clean_text"])
        joblib.dump(vec, vec_path)
    Xva = vec.transform(va["clean_text"])
    Xte = vec.transform(te["clean_text"])
    ytr, yte = tr["label"].values, te["label"].values

    # ----- Logistic Regression -----
    lr = LogisticRegression(**config.LR)
    t0 = time.time(); lr.fit(Xtr, ytr); print(f"  LR fit: {time.time()-t0:.1f}s")
    pred_lr = lr.predict(Xte)
    joblib.dump(lr, MODEL_DIR / f"lr_{task}.joblib")
    pd.DataFrame({"y_true": yte, "y_pred": pred_lr}).to_csv(
        RESULTS_DIR / f"pred_{task}_lr.csv", index=False
    )

    # ----- Linear SVM -----
    svm_base = LinearSVC(**config.SVC_PARAMS)
    svm = CalibratedClassifierCV(svm_base, cv=3, method="sigmoid", n_jobs=-1)
    t0 = time.time(); svm.fit(Xtr, ytr); print(f"  SVM fit: {time.time()-t0:.1f}s")
    pred_svm = svm.predict(Xte)
    joblib.dump(svm, MODEL_DIR / f"svm_{task}.joblib")
    pd.DataFrame({"y_true": yte, "y_pred": pred_svm}).to_csv(
        RESULTS_DIR / f"pred_{task}_svm.csv", index=False
    )

    from sklearn.metrics import f1_score, accuracy_score
    for name, p in [("LR", pred_lr), ("SVM", pred_svm)]:
        print(f"  {name:>3}  acc={accuracy_score(yte, p):.4f}  "
              f"macro-F1={f1_score(yte, p, average='macro'):.4f}  "
              f"weighted-F1={f1_score(yte, p, average='weighted'):.4f}")

    return vec


In [8]:
vec_sent = run_classical("sentiment")
vec_emo  = run_classical("emotion")



=== SENTIMENT — TF-IDF + LR/SVM ===
  LR fit: 176.2s
  SVM fit: 14.7s
   LR  acc=0.6598  macro-F1=0.3852  weighted-F1=0.7107
  SVM  acc=0.8783  macro-F1=0.4004  weighted-F1=0.8409

=== EMOTION — TF-IDF + LR/SVM ===
  LR fit: 335.9s
  SVM fit: 33.8s
   LR  acc=0.3601  macro-F1=0.1632  weighted-F1=0.4959
  SVM  acc=0.8773  macro-F1=0.2087  weighted-F1=0.8347


## 5 — fastText Urdu embeddings download

The CNN and BiLSTM are initialised from the publicly released fastText
`cc.ur.300` vectors (Facebook AI). The file is ≈ 600 MB compressed; cached.


In [9]:
from urllib.request import urlretrieve
if not config.FASTTEXT_PATH.exists():
    print(f"Downloading fastText Urdu vectors to {config.FASTTEXT_PATH} ...")
    urlretrieve(config.FASTTEXT_URL, config.FASTTEXT_PATH)
    print("  done.")
else:
    print(f"Found fastText vectors at {config.FASTTEXT_PATH} "
          f"({config.FASTTEXT_PATH.stat().st_size/1e6:.1f} MB)")


Found fastText vectors at C:\Users\USER\Desktop\NLP Project\Assignment#03\outputs\embeddings\cc.ur.300.vec.gz (679.2 MB)


## 6 — Deep Learning: CNN + BiLSTM (PyTorch)


In [10]:
from torch.utils.data import DataLoader
import train_dl as tdl
from train_dl import Vocab, TweetDataset, make_collate, load_fasttext_vectors, compute_class_weights
from models_dl import TextCNN, BiLSTMAttn

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def build_dataloaders(task: str, batch_size: int):
    tr, va, te = splits[task]["train"], splits[task]["val"], splits[task]["test"]
    vocab_path = CACHE_DIR / f"vocab_{task}.json"
    if vocab_path.exists():
        vocab = Vocab.from_json(vocab_path)
    else:
        vocab = Vocab.build(tr["clean_text"], max_size=config.VOCAB_SIZE, min_freq=2)
        vocab.to_json(vocab_path)
    print(f"  {task} vocab: {len(vocab):,} tokens (pad_id={vocab.pad_id}, unk_id={vocab.unk_id})")

    collate = make_collate(vocab.pad_id, config.MAX_LEN)
    def loader(df, shuffle):
        ds = TweetDataset(df["clean_text"].tolist(), df["label"].tolist(), vocab, config.MAX_LEN)
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=2,
                          pin_memory=True, collate_fn=collate)
    return vocab, loader(tr, True), loader(va, False), loader(te, False)

def fasttext_for(vocab: Vocab):
    emb_path = EMBED_DIR / f"ft_matrix_{len(vocab)}.pt"
    if emb_path.exists():
        return torch.load(emb_path)
    E = load_fasttext_vectors(config.FASTTEXT_PATH, vocab, dim=config.EMBED_DIM)
    torch.save(E, emb_path)
    return E

def run_dl(task: str, model_name: str):
    num_classes = 3 if task == "sentiment" else 6
    cfg = config.CNN if model_name == "cnn" else config.BILSTM
    print(f"\n=== {task.upper()} — {model_name.upper()} ===")

    vocab, tl, vl, tel = build_dataloaders(task, cfg["batch_size"])
    E = fasttext_for(vocab)
    if model_name == "cnn":
        model = TextCNN(
            vocab_size=len(vocab), num_classes=num_classes,
            embed_dim=config.EMBED_DIM,
            filter_sizes=cfg["filter_sizes"], num_filters=cfg["num_filters"],
            dropout=cfg["dropout"], padding_idx=vocab.pad_id,
            pretrained_embeddings=E,
        )
        needs_mask = False
    else:
        model = BiLSTMAttn(
            vocab_size=len(vocab), num_classes=num_classes,
            embed_dim=config.EMBED_DIM,
            hidden_size=cfg["hidden_size"], num_layers=cfg["num_layers"],
            dropout=cfg["dropout"], padding_idx=vocab.pad_id,
            pretrained_embeddings=E,
        )
        needs_mask = True
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  trainable params: {n_params:,}")

    ytr_arr = np.asarray(splits[task]["train"]["label"].values, dtype=np.int64)
    cw = compute_class_weights(ytr_arr, num_classes)
    print(f"  class weights: {cw.tolist()}")

    out = tdl.train_model(
        model, tl, vl, class_weights=cw,
        lr=cfg["lr"], epochs=cfg["epochs"], patience=cfg["patience"],
        device=DEVICE, needs_mask=needs_mask,
    )

    y_true, y_pred = tdl.predict(model, tel, DEVICE, needs_mask)
    pd.DataFrame({"y_true": y_true, "y_pred": y_pred}).to_csv(
        RESULTS_DIR / f"pred_{task}_{model_name}.csv", index=False
    )
    torch.save(model.state_dict(), MODEL_DIR / f"{model_name}_{task}.pt")
    with open(MODEL_DIR / f"{model_name}_{task}.history.json", "w", encoding="utf-8") as f:
        json.dump({"history": out["history"], "best_val_acc": out["best_val_acc"]}, f)

    from sklearn.metrics import f1_score, accuracy_score
    print(f"  TEST  acc={accuracy_score(y_true, y_pred):.4f}  "
          f"macro-F1={f1_score(y_true, y_pred, average='macro'):.4f}  "
          f"weighted-F1={f1_score(y_true, y_pred, average='weighted'):.4f}")

    del model, tl, vl, tel
    torch.cuda.empty_cache(); gc.collect()


In [11]:
run_dl("sentiment", "cnn")
run_dl("sentiment", "bilstm")



=== SENTIMENT — CNN ===
  sentiment vocab: 58,851 tokens (pad_id=0, unk_id=1)
  trainable params: 18,117,639
  class weights: [2.4632327556610107, 114.76207733154297, 0.38679996132850647]
  epoch  1  step   100  loss 1.0700
  epoch  1  step   200  loss 1.0455
  epoch  1  step   300  loss 1.0289
  epoch  1  step   400  loss 1.0174
  epoch  1  step   500  loss 1.0132
  epoch  1  step   600  loss 1.0124
  epoch  1  step   700  loss 1.0053
  epoch  1  step   800  loss 0.9984
  epoch  1  step   900  loss 0.9954
  epoch  1  step  1000  loss 0.9876
  epoch  1  step  1100  loss 0.9830
  epoch  1  step  1200  loss 0.9804
  epoch  1  step  1300  loss 0.9802
  epoch  1  step  1400  loss 0.9787
  epoch  1 done  (24.4s)   train_loss=0.9771   val_acc=0.5620
  epoch  2  step   100  loss 0.7733
  epoch  2  step   200  loss 0.7710
  epoch  2  step   300  loss 0.7672
  epoch  2  step   400  loss 0.7574
  epoch  2  step   500  loss 0.7583
  epoch  2  step   600  loss 0.7487
  epoch  2  step   700  loss 

In [12]:
run_dl("emotion", "cnn")
run_dl("emotion", "bilstm")



=== EMOTION — CNN ===
  emotion vocab: 58,803 tokens (pad_id=0, unk_id=1)
  trainable params: 18,104,394
  class weights: [0.19339998066425323, 1.7623409032821655, 31.722137451171875, 48.32322311401367, 5.2003068923950195, 57.381038665771484]
  epoch  1  step   100  loss 1.7775
  epoch  1  step   200  loss 1.7671
  epoch  1  step   300  loss 1.7481
  epoch  1  step   400  loss 1.7310
  epoch  1  step   500  loss 1.7159
  epoch  1  step   600  loss 1.7089
  epoch  1  step   700  loss 1.6996
  epoch  1  step   800  loss 1.6905
  epoch  1  step   900  loss 1.6840
  epoch  1  step  1000  loss 1.6731
  epoch  1  step  1100  loss 1.6641
  epoch  1  step  1200  loss 1.6577
  epoch  1  step  1300  loss 1.6511
  epoch  1  step  1400  loss 1.6459
  epoch  1 done  (14.6s)   train_loss=1.6427   val_acc=0.4690
  epoch  2  step   100  loss 1.2911
  epoch  2  step   200  loss 1.2642
  epoch  2  step   300  loss 1.2607
  epoch  2  step   400  loss 1.2656
  epoch  2  step   500  loss 1.2765
  epoch  2

## 7 — Transformer fine-tuning: mBERT, XLM-RoBERTa, Urdu-RoBERTa

HuggingFace Trainer with class-weighted cross-entropy, fp16, and the
standard warm-up + linear-decay schedule (§6.3 of the math doc).


In [8]:
from train_transformer import TransformerCfg, finetune_transformer

def transformer_class_weights(labels_arr, num_classes):
    from sklearn.utils.class_weight import compute_class_weight
    cls = np.arange(num_classes)
    return compute_class_weight(class_weight="balanced", classes=cls, y=labels_arr).tolist()

def run_transformer(task: str, model_key: str):
    cfg_dict = config.TRANSFORMERS[model_key]
    cfg = TransformerCfg(name=model_key, max_len=config.TRANSFORMER_MAX_LEN, **cfg_dict)
    tr, va, te = splits[task]["train"], splits[task]["val"], splits[task]["test"]
    if TRANSFORMER_TRAIN_SUBSET:
        tr = tr.groupby("label", group_keys=False).apply(
            lambda g: g.sample(min(len(g), max(1, int(len(g)*TRANSFORMER_TRAIN_SUBSET/len(tr)))),
                               random_state=SEED)
        ).reset_index(drop=True)

    if task == "sentiment":
        id2label = {i: l for i, l in enumerate(["Negative", "Neutral", "Positive"])}
    else:
        id2label = {i: l for i, l in enumerate(EMOTION_LABELS)}
    label2id = {v: k for k, v in id2label.items()}

    cw = transformer_class_weights(tr["label"].values, num_classes=len(id2label))
    print(f"  {task}/{model_key}: train={len(tr):,}, weights={cw}")

    res = finetune_transformer(
        cfg=cfg, train_df=tr, val_df=va, test_df=te,
        text_col="clean_text", label_col="label",
        id2label=id2label, label2id=label2id,
        class_weights=cw, output_root=MODEL_DIR,
        task_name=task,
    )
    # Save predictions in the standard location too:
    preds = pd.read_csv(Path(res["output_dir"]) / "test_predictions.csv")
    preds.to_csv(RESULTS_DIR / f"pred_{task}_{model_key}.csv", index=False)
    return res


In [14]:
# Sentiment task
run_transformer("sentiment", "mbert")


  sentiment/mbert: train=372,862, weights=[2.4632327196094366, 114.76208064019698, 0.38679994937580786]


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 79900/79900 [00:01<00:00, 61229.83 examples/s]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.872300,1.011369,0.782776,0.429231,0.803985
2,0.739400,1.023168,0.776055,0.430249,0.800410
3,0.798200,1.031615,0.804265,0.446387,0.820651


{'preds': array([2, 2, 2, ..., 2, 2, 2], shape=(79900,)),
 'metrics': {'task': 'sentiment',
  'model': 'mbert',
  'val': {'eval_loss': 1.031615138053894,
   'eval_accuracy': 0.8042653850486239,
   'eval_f1_macro': 0.4463866159547161,
   'eval_f1_weighted': 0.8206512532091891,
   'eval_runtime': 29.5797,
   'eval_samples_per_second': 2701.142,
   'eval_steps_per_second': 42.225,
   'epoch': 3.0},
  'test': {'test_loss': 1.0111902952194214,
   'test_accuracy': 0.8054443053817272,
   'test_f1_macro': 0.4526474913567841,
   'test_f1_weighted': 0.8216717045885426,
   'test_runtime': 30.2856,
   'test_samples_per_second': 2638.222,
   'test_steps_per_second': 41.241}},
 'output_dir': 'C:\\Users\\USER\\Desktop\\NLP Project\\Assignment#03\\outputs\\models\\mbert_sentiment'}

In [9]:
run_transformer("sentiment", "xlm-r")


  sentiment/xlm-r: train=372,862, weights=[2.4632327196094366, 114.76208064019698, 0.38679994937580786]


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 79900/79900 [00:00<00:00, 97068.77 examples/s] 


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.891200,0.981296,0.792926,0.431906,0.810571
2,0.726600,1.043206,0.726092,0.411075,0.764296
3,0.844600,1.070329,0.785216,0.438254,0.808067


{'preds': array([0, 2, 2, ..., 0, 2, 0], shape=(79900,)),
 'metrics': {'task': 'sentiment',
  'model': 'xlm-r',
  'val': {'eval_loss': 1.070328712463379,
   'eval_accuracy': 0.785216335623725,
   'eval_f1_macro': 0.438253518163698,
   'eval_f1_weighted': 0.8080666145212567,
   'eval_runtime': 27.1832,
   'eval_samples_per_second': 2939.279,
   'eval_steps_per_second': 45.947,
   'epoch': 3.0},
  'test': {'test_loss': 1.0338211059570312,
   'test_accuracy': 0.7896620775969962,
   'test_f1_macro': 0.4474681701307377,
   'test_f1_weighted': 0.8118390687494448,
   'test_runtime': 28.2338,
   'test_samples_per_second': 2829.937,
   'test_steps_per_second': 44.238}},
 'output_dir': 'C:\\Users\\USER\\Desktop\\NLP Project\\Assignment#03\\outputs\\models\\xlm-r_sentiment'}

In [10]:
run_transformer("sentiment", "urdu-roberta")


  sentiment/urdu-roberta: train=372,862, weights=[2.4632327196094366, 114.76208064019698, 0.38679994937580786]


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at urduhack/roberta-urdu-small and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 79900/79900 [00:00<00:00, 89126.09 examples/s] 


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,1.000000,1.158882,0.761699,0.423153,0.789868
2,0.801300,1.282096,0.745579,0.422086,0.778911
3,0.829400,1.257526,0.773176,0.440827,0.799523


{'preds': array([0, 2, 2, ..., 0, 2, 2], shape=(79900,)),
 'metrics': {'task': 'sentiment',
  'model': 'urdu-roberta',
  'val': {'eval_loss': 1.257526159286499,
   'eval_accuracy': 0.7731761348702737,
   'eval_f1_macro': 0.4408270609146778,
   'eval_f1_weighted': 0.7995229494359467,
   'eval_runtime': 28.3244,
   'eval_samples_per_second': 2820.851,
   'eval_steps_per_second': 22.066,
   'epoch': 3.0},
  'test': {'test_loss': 1.2419825792312622,
   'test_accuracy': 0.7750187734668336,
   'test_f1_macro': 0.45725950008236116,
   'test_f1_weighted': 0.8010672089604743,
   'test_runtime': 28.724,
   'test_samples_per_second': 2781.647,
   'test_steps_per_second': 21.759}},
 'output_dir': 'C:\\Users\\USER\\Desktop\\NLP Project\\Assignment#03\\outputs\\models\\urdu-roberta_sentiment'}

In [11]:
# Emotion task
run_transformer("emotion", "mbert")


  emotion/mbert: train=372,862, weights=[0.19339997468790393, 1.7623409524889873, 31.72213714480177, 48.323224468636596, 5.200306834030683, 57.38104032009849]


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 79900/79900 [00:01<00:00, 69261.52 examples/s]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,1.532300,1.765515,0.703563,0.224434,0.748877
2,1.510100,1.708905,0.668056,0.247715,0.728725
3,1.343800,1.760656,0.690710,0.260641,0.746018


{'preds': array([0, 4, 0, ..., 0, 0, 1], shape=(79900,)),
 'metrics': {'task': 'emotion',
  'model': 'mbert',
  'val': {'eval_loss': 1.7606559991836548,
   'eval_accuracy': 0.690709520769972,
   'eval_f1_macro': 0.26064096694905237,
   'eval_f1_weighted': 0.7460175522239072,
   'eval_runtime': 31.0912,
   'eval_samples_per_second': 2569.83,
   'eval_steps_per_second': 40.172,
   'epoch': 3.0},
  'test': {'test_loss': 1.7260329723358154,
   'test_accuracy': 0.6922403003754694,
   'test_f1_macro': 0.2703445759299271,
   'test_f1_weighted': 0.7467129005295166,
   'test_runtime': 32.4366,
   'test_samples_per_second': 2463.266,
   'test_steps_per_second': 38.506}},
 'output_dir': 'C:\\Users\\USER\\Desktop\\NLP Project\\Assignment#03\\outputs\\models\\mbert_emotion'}

In [12]:
run_transformer("emotion", "xlm-r")


  emotion/xlm-r: train=372,862, weights=[0.19339997468790393, 1.7623409524889873, 31.72213714480177, 48.323224468636596, 5.200306834030683, 57.38104032009849]


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 79900/79900 [00:01<00:00, 50210.88 examples/s]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,1.524800,1.753492,0.722199,0.231649,0.767122
2,1.545300,1.740492,0.649495,0.237641,0.715245
3,1.451100,1.764959,0.689658,0.253501,0.745965


{'preds': array([0, 0, 0, ..., 0, 0, 1], shape=(79900,)),
 'metrics': {'task': 'emotion',
  'model': 'xlm-r',
  'val': {'eval_loss': 1.7649590969085693,
   'eval_accuracy': 0.6896581934692549,
   'eval_f1_macro': 0.25350118201590016,
   'eval_f1_weighted': 0.7459648560797835,
   'eval_runtime': 27.6409,
   'eval_samples_per_second': 2890.611,
   'eval_steps_per_second': 45.187,
   'epoch': 3.0},
  'test': {'test_loss': 1.7508302927017212,
   'test_accuracy': 0.6906633291614518,
   'test_f1_macro': 0.25353298028217147,
   'test_f1_weighted': 0.7461967712177684,
   'test_runtime': 28.1583,
   'test_samples_per_second': 2837.525,
   'test_steps_per_second': 44.356}},
 'output_dir': 'C:\\Users\\USER\\Desktop\\NLP Project\\Assignment#03\\outputs\\models\\xlm-r_emotion'}

In [13]:
run_transformer("emotion", "urdu-roberta")


  emotion/urdu-roberta: train=372,862, weights=[0.19339997468790393, 1.7623409524889873, 31.72213714480177, 48.323224468636596, 5.200306834030683, 57.38104032009849]


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at urduhack/roberta-urdu-small and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 79900/79900 [00:00<00:00, 91134.03 examples/s] 


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,1.635700,1.790798,0.596728,0.205297,0.665200
2,1.526000,1.709621,0.621710,0.237884,0.699628
3,1.286000,1.841284,0.613988,0.247714,0.695840


{'preds': array([0, 0, 0, ..., 0, 0, 4], shape=(79900,)),
 'metrics': {'task': 'emotion',
  'model': 'urdu-roberta',
  'val': {'eval_loss': 1.8412840366363525,
   'eval_accuracy': 0.6139876594200178,
   'eval_f1_macro': 0.2477142321532995,
   'eval_f1_weighted': 0.6958404332997518,
   'eval_runtime': 30.0859,
   'eval_samples_per_second': 2655.695,
   'eval_steps_per_second': 20.774,
   'epoch': 3.0},
  'test': {'test_loss': 1.7951668500900269,
   'test_accuracy': 0.6153441802252816,
   'test_f1_macro': 0.25393742918887724,
   'test_f1_weighted': 0.6971450817608247,
   'test_runtime': 30.448,
   'test_samples_per_second': 2624.149,
   'test_steps_per_second': 20.527}},
 'output_dir': 'C:\\Users\\USER\\Desktop\\NLP Project\\Assignment#03\\outputs\\models\\urdu-roberta_emotion'}

## 8 — Inference demo on hand-crafted tweets


In [14]:
import joblib
from transformers import AutoModelForSequenceClassification, AutoTokenizer

demo = [
    "آج کا دن بہت اچھا تھا یہ واقعی خوبصورت لمحہ ہے",         # positive / joy
    "حکومت کی پالیسیاں بالکل غلط ہیں کوئی بہتری نہیں آئی",     # negative / anger
    "یہ خبر سن کر بہت حیرت ہوئی واقعی عجیب بات ہے",            # neutral / surprise
    "والد کی محبت سے زیادہ کوئی نعمت نہیں",                    # positive / love
    "ہمارا ساتھی شدید بیمار ہو گیا ہے اللہ کرم کرے",            # negative / fear or sadness
]
print("Demo tweets (cleaned):")
for t in demo:
    print("  ", pp.preprocess_tweet(t))


Demo tweets (cleaned):
   آج کا دن بہت اچھا تھا یہ واقعی خوبصورت لمحہ ہے
   حکومت کی پالیسیاں بالکل غلط ہیں کوئی بہتری نہیں آئی
   یہ خبر سن کر بہت حیرت ہوئی واقعی عجیب بات ہے
   والد کی محبت سے زیادہ کوئی نعمت نہیں
   ہمارا ساتھی شدید بیمار ہو گیا ہے اللہ کرم کرے


In [15]:
# Pick the best transformer per task by val macro-F1 logged in metrics.json
import glob
def pick_best(task):
    cands = []
    for d in glob.glob(str(MODEL_DIR / f"*_{task}")):
        mp = Path(d) / "metrics.json"
        if mp.exists():
            m = json.loads(mp.read_text(encoding="utf-8"))
            cands.append((m["val"].get("eval_f1_macro", 0), d, m["model"]))
    cands.sort(reverse=True)
    return cands[0] if cands else None

for task in ("sentiment", "emotion"):
    pick = pick_best(task)
    if pick is None:
        print(f"  no transformer trained yet for {task}")
        continue
    score, dpath, mname = pick
    print(f"\nBest transformer for {task}: {mname} (val_f1_macro={score:.4f})  -> {dpath}")
    tok = AutoTokenizer.from_pretrained(str(Path(dpath) / "best"))
    mdl = AutoModelForSequenceClassification.from_pretrained(str(Path(dpath) / "best"))
    mdl.eval()
    with torch.no_grad():
        for t in demo:
            inp = tok([pp.preprocess_tweet(t)], padding=True, truncation=True,
                      max_length=config.TRANSFORMER_MAX_LEN, return_tensors="pt")
            pred_id = int(mdl(**inp).logits.argmax(-1).item())
            lab = mdl.config.id2label[pred_id]
            print(f"  [{lab:>10}]   {t}")


The tokenizer you are loading from 'C:\Users\USER\Desktop\NLP Project\Assignment#03\outputs\models\mbert_sentiment\best' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.



Best transformer for sentiment: mbert (val_f1_macro=0.4464)  -> C:\Users\USER\Desktop\NLP Project\Assignment#03\outputs\models\mbert_sentiment
  [  Positive]   آج کا دن بہت اچھا تھا یہ واقعی خوبصورت لمحہ ہے
  [  Positive]   حکومت کی پالیسیاں بالکل غلط ہیں کوئی بہتری نہیں آئی
  [  Negative]   یہ خبر سن کر بہت حیرت ہوئی واقعی عجیب بات ہے
  [  Positive]   والد کی محبت سے زیادہ کوئی نعمت نہیں
  [  Negative]   ہمارا ساتھی شدید بیمار ہو گیا ہے اللہ کرم کرے

Best transformer for emotion: mbert (val_f1_macro=0.2606)  -> C:\Users\USER\Desktop\NLP Project\Assignment#03\outputs\models\mbert_emotion


The tokenizer you are loading from 'C:\Users\USER\Desktop\NLP Project\Assignment#03\outputs\models\mbert_emotion\best' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


  [       Sad]   آج کا دن بہت اچھا تھا یہ واقعی خوبصورت لمحہ ہے
  [       Sad]   حکومت کی پالیسیاں بالکل غلط ہیں کوئی بہتری نہیں آئی
  [       Sad]   یہ خبر سن کر بہت حیرت ہوئی واقعی عجیب بات ہے
  [   Disgust]   والد کی محبت سے زیادہ کوئی نعمت نہیں
  [       Sad]   ہمارا ساتھی شدید بیمار ہو گیا ہے اللہ کرم کرے


## 9 — Summary of where everything was saved


In [16]:
print("Saved test predictions in", RESULTS_DIR)
for p in sorted(RESULTS_DIR.glob("pred_*.csv")):
    print("  ", p.name)
print("\nSaved models in", MODEL_DIR)
for p in sorted(MODEL_DIR.iterdir()):
    print("  ", p.name)


Saved test predictions in C:\Users\USER\Desktop\NLP Project\Assignment#03\outputs\results
   pred_emotion_bilstm.csv
   pred_emotion_cnn.csv
   pred_emotion_lr.csv
   pred_emotion_mbert.csv
   pred_emotion_svm.csv
   pred_emotion_urdu-roberta.csv
   pred_emotion_xlm-r.csv
   pred_sentiment_bilstm.csv
   pred_sentiment_cnn.csv
   pred_sentiment_lr.csv
   pred_sentiment_mbert.csv
   pred_sentiment_svm.csv
   pred_sentiment_urdu-roberta.csv
   pred_sentiment_xlm-r.csv

Saved models in C:\Users\USER\Desktop\NLP Project\Assignment#03\outputs\models
   bilstm_emotion.history.json
   bilstm_emotion.pt
   bilstm_sentiment.history.json
   bilstm_sentiment.pt
   cnn_emotion.history.json
   cnn_emotion.pt
   cnn_sentiment.history.json
   cnn_sentiment.pt
   lr_emotion.joblib
   lr_sentiment.joblib
   mbert_emotion
   mbert_sentiment
   svm_emotion.joblib
   svm_sentiment.joblib
   tfidf_emotion.joblib
   tfidf_sentiment.joblib
   urdu-roberta_emotion
   urdu-roberta_sentiment
   xlm-r_emotion
   